In [ ]:
!pip install momentfm --no-deps -q
!pip install einops -q

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from momentfm import MOMENTPipeline
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
X_tr = torch.tensor(data['train']['X'], dtype=torch.float32)
X_te = torch.tensor(data['test']['X'], dtype=torch.float32)

le = LabelEncoder()
y_tr = torch.tensor(le.fit_transform(data['train']['y']), dtype=torch.long)
y_te = torch.tensor(le.transform(data['test']['y']), dtype=torch.long)

print(f'CHB-MIT: X_train={X_tr.shape}, X_test={X_te.shape}, classes={le.classes_}')

In [ ]:
model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large',
    model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
)
model.init()
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
optimizer = Adam(model.head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(x_enc=X_batch)
        loss = criterion(out.logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

torch.save(model.state_dict(), f'{SAVE_DIR}/MOMENT_CHBMIT_v2_trained.pt')
print('Saved!')

model.eval()
all_preds = []
with torch.no_grad():
    for i in range(0, len(X_te), 32):
        X_batch = X_te[i:i+32].to(device)
        out = model(x_enc=X_batch)
        preds = out.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

print(accuracy_score(y_te.numpy(), all_preds))
print(classification_report(y_te.numpy(), all_preds))

In [ ]:
from openpyxl import load_workbook

XLSX = "/content/drive/MyDrive/results.xlsx"
wb = load_workbook(XLSX)
ws = wb["Accuracy"]

for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    if row[0].value == "MOMENT":
        row[7].value = 0.8257  # CHB-MIT column
        print("updated MOMENT CHB-MIT cell to 0.8257")
        break

wb.save(XLSX)

In [ ]:
te = data['test']
y_te_int = le.transform(te['y'])
file = np.asarray(te['file'])
t0 = te['t0'].astype(int)

MERGE = 30
event_id = np.full(len(y_te_int), -1, dtype=int)
next_ev = 0
for f in np.unique(file):
    idx = np.where((file == f) & (y_te_int == 1))[0]
    if len(idx) == 0:
        continue
    order = idx[np.argsort(t0[idx])]
    starts = t0[order]
    ev_local = np.zeros(len(order), dtype=int)
    for i in range(1, len(order)):
        ev_local[i] = ev_local[i-1] + (1 if (starts[i] - starts[i-1]) > MERGE else 0)
    for local, gi in zip(ev_local, order):
        event_id[gi] = next_ev + local
    next_ev += int(ev_local.max()) + 1

n_events = next_ev
neg_mask = (y_te_int == 0)
print("ground-truth seizure events:", n_events)

def sens_at_fpr(y_score, fpr_targets=(0.05, 0.10, 0.20)):
    rows = []
    for thr in np.unique(y_score):
        pred = y_score >= thr
        det = np.array([pred[event_id == ev].any() for ev in range(n_events)])
        rows.append((thr, det.mean(), pred[neg_mask].mean()))
    rows = np.array(rows)
    out = {}
    for tgt in fpr_targets:
        ok = rows[rows[:, 2] <= tgt]
        out[tgt] = ok[:, 1].max() if len(ok) else np.nan
    return out, rows

@torch.no_grad()
def moment_scores(X):
    model.eval()
    all_scores = []
    for i in range(0, len(X), 32):
        xb = X[i:i+32].to(device)
        out = model(x_enc=xb)
        score = (out.logits[:, 1] - out.logits[:, 0]).cpu().numpy()
        all_scores.extend(score)
    return np.array(all_scores)

In [ ]:
def permute_channels(X, seed):
    r = np.random.default_rng(seed)
    out = X.clone()
    for w in range(out.shape[0]):
        perm = torch.tensor(r.permutation(out.shape[1]))
        out[w] = out[w, perm]
    return out

def desync_channels(X, seed):
    r = np.random.default_rng(seed)
    out = X.clone().numpy()
    for w in range(out.shape[0]):
        for c in range(out.shape[1]):
            out[w, c] = np.roll(out[w, c], r.integers(0, out.shape[2]))
    return torch.tensor(out, dtype=torch.float32)

def pool_channels_broadcast(X):
    mean_signal = X.mean(dim=1, keepdim=True)
    return mean_signal.repeat(1, X.shape[1], 1)

In [ ]:
prob_base = moment_scores(X_te)
res_base, _ = sens_at_fpr(prob_base)
print("MOMENT baseline event-level sensitivity:")
for tgt, s in res_base.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

X_te_perm = permute_channels(X_te, seed=0)
prob_perm = moment_scores(X_te_perm)
res_perm, _ = sens_at_fpr(prob_perm)
print("\nMOMENT permuted-channels event-level sensitivity:")
for tgt, s in res_perm.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

X_te_desync = desync_channels(X_te, seed=0)
prob_desync = moment_scores(X_te_desync)
res_desync, _ = sens_at_fpr(prob_desync)
print("\nMOMENT desynced-channels event-level sensitivity:")
for tgt, s in res_desync.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

X_te_pool = pool_channels_broadcast(X_te)
prob_pool = moment_scores(X_te_pool)
res_pool, _ = sens_at_fpr(prob_pool)
print("\nMOMENT channel-pooled event-level sensitivity:")
for tgt, s in res_pool.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font

XLSX = "/content/drive/MyDrive/results.xlsx"
wb = load_workbook(XLSX)
bold = Font(bold=True)
sheet_name = "RQ1 MOMENT ablations"
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

ws.cell(row=1, column=1, value="RQ1/RQ3: MOMENT (fine-tuned) channel ablations, event-level sensitivity at fixed interictal FPR (CHB-MIT v2, 34 events)").font = bold
headers = ["Ablation", "Sens @5% FPR", "Sens @10% FPR", "Sens @20% FPR"]
for c, h in enumerate(headers, 1):
    ws.cell(row=3, column=c, value=h).font = bold

rows = [
    ("Baseline (unmodified)", res_base[0.05], res_base[0.10], res_base[0.20]),
    ("Permuted channel order", res_perm[0.05], res_perm[0.10], res_perm[0.20]),
    ("Desynced channel timing", res_desync[0.05], res_desync[0.10], res_desync[0.20]),
    ("Channel-pooled (broadcast)", res_pool[0.05], res_pool[0.10], res_pool[0.20]),
]

r = 4
for row in rows:
    for c, v in enumerate(row, 1):
        cell = ws.cell(row=r, column=c, value=v)
        if c > 1:
            cell.number_format = "0.000"
    r += 1

ws.cell(row=r+1, column=1, value="Note: MOMENT was fine-tuned (head only) on CHB-MIT, unlike Chronos-2/MantisV2 which used frozen probes. The same channel-blindness pattern held regardless.")

wb.save(XLSX)
print("saved")